# 3. Exploratory data analysis using SQL
Use SQLite to answer data questions with SQL. The notebook creates a table from the available course CSV.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists(): ROOT = ROOT.parent
DATA = ROOT / "data"
file = next((DATA / n for n in ["dataset_part_2.csv", "dataset_part_1.csv", "spacex_launch_dash.csv"] if (DATA/n).exists()), None)
if file is None: raise FileNotFoundError("Run src/download_data.py first.")
df = pd.read_csv(file)
# SQL identifiers are normalized to safe lowercase underscores.
df.columns = [str(c).strip().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "").replace("/", "_") for c in df.columns]
con = sqlite3.connect(":memory:")
df.to_sql("launches", con, index=False, if_exists="replace")
print("Using", file.name, "| rows:", len(df))
display(pd.read_sql_query("SELECT * FROM launches LIMIT 5", con))

In [ ]:
# Q1: How many rows are in the dataset?
display(pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM launches", con))

In [ ]:
# Q2: Which columns are available? (SQLite schema)
display(pd.read_sql_query("PRAGMA table_info(launches)", con))

In [ ]:
# Q3: Summarize launch sites when the dataset includes that column.
cols = set(df.columns)
site = next((c for c in ["Launch_Site", "LaunchSite", "launch_site"] if c in cols), None)
if site:
    query = f'SELECT "{site}" AS launch_site, COUNT(*) AS launches FROM launches GROUP BY "{site}" ORDER BY launches DESC'
    display(pd.read_sql_query(query, con))
else:
    print("No launch-site column found in this dataset version.")

In [ ]:
# Q4: Inspect the target/outcome distribution when available.
target = next((c for c in ["Class", "class", "LandingOutcome", "Outcome"] if c in cols), None)
if target:
    query = f'SELECT "{target}" AS outcome, COUNT(*) AS count FROM launches GROUP BY "{target}" ORDER BY count DESC'
    display(pd.read_sql_query(query, con))
else:
    print("No target/outcome column found in this dataset version.")